In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
self_np = np # Alias for eval context
from scipy.integrate import solve_ivp
import re
import z3
import random
from typing import Dict, Any, Tuple, List, Optional, Union

from pyast_config import (
    ActionType, ChoiceActionConfig, SequenceActionConfig, 
    DeterministicAssignmentActionConfig, ActionConstraintConfig, 
    DefinitionsConfig, ConstantDefinitionConfig, ControlFuncConfig, 
    StepFuncConfig, ProblemConfig, PreconditionConfig, 
    PostconditionConfig, LoopExecutionConfig
)

class GeneratedKXEnv(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        self._setup_configs()
        
        self._state_keys = ['x', 'y', 'v', 'a', 'dx', 'dy', 'w', 'r', 'xo', 'yo', 't']
        self.state = {k: 0.0 for k in self._state_keys}
        self.constants = {c.name: float(c.value) for c in self._definitions_config.constants}
        
        # Observation Space: 11 base keys + 2 relative distances = 13
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, 
            shape=(len(self._state_keys) + 2,), 
            dtype=np.float32
        )

        self.action_space = spaces.Dict({
            'choice_ChoiceLevel_0': spaces.Discrete(3),
            'assign_w': spaces.Box(-1.0, 1.0, shape=(), dtype=np.float32),
            'assign_r': spaces.Box(1.0, 10.0, shape=(), dtype=np.float32),
            'assign_xo': spaces.Box(-20, 20, shape=(), dtype=np.float32),
            'assign_yo': spaces.Box(-20, 20, shape=(), dtype=np.float32),
        })

        self.functions = {
            'stopDist': lambda v: v**2 / (2*self.constants['b']),
            'accelComp': lambda v: (self.constants['A']/self.constants['b'] + 1) * (self.constants['A']/2 * self.constants['ep']**2 + self.constants['ep']*v),
            'admissibleSeparation': lambda v: self.functions['stopDist'](v) + self.functions['accelComp'](v)
        }

        self.max_episode_steps = 100

    def _setup_configs(self):
        self._definitions_config = DefinitionsConfig(
            constants=[
                ConstantDefinitionConfig('ep', 'Real', '0.1', {}),
                ConstantDefinitionConfig('b', 'Real', '2.0', {}),
                ConstantDefinitionConfig('A', 'Real', '5.0', {}),
                ConstantDefinitionConfig('W', 'Real', '1.0', {})
            ],
            functions=[], predicates=[],
            hybrid_programs=[
                ControlFuncConfig('controlFunc', ChoiceActionConfig('ChoiceLevel_0', [
                    SequenceActionConfig([DeterministicAssignmentActionConfig('a', '-b')]), # 0: Brake
                    SequenceActionConfig([ActionConstraintConfig('v == 0'), DeterministicAssignmentActionConfig('a', '0')]), # 1: Stand
                    SequenceActionConfig([DeterministicAssignmentActionConfig('a', 'A'), 
                                         ActionConstraintConfig('abs(x-xo) > admissibleSeparation(v) | abs(y-yo) > admissibleSeparation(v)')]) # 2: Accel
                ]), {}),
                StepFuncConfig('stepFunc', [], None, {})
            ]
        )

    def step(self, action):
        applied_reward, term, applied = self._execute_action_config(
            self._definitions_config.hybrid_programs[0].action_space_config, 
            action
        )
        
        if not applied:
            return self._get_obs(), -5.0, False, False, self._get_info()

        # Physics Integration
        y0 = [self.state[k] for k in self._state_keys]
        sol = solve_ivp(self._ode_func, (0, self.constants['ep']), y0, method='RK45')
        for i, k in enumerate(self._state_keys): 
            self.state[k] = float(sol.y[i, -1])
        
        if self.state['v'] < 0: self.state['v'] = 0.0

        is_safe = self._check_safety()
        if is_safe:
            if self.state['v'] > 0.1:
                reward = 0.5 + (self.state['v'] * 1.5) 
            else:
                reward = -1.0 
            
            dist = np.sqrt((self.state['x'] - self.state['xo'])**2 + (self.state['y'] - self.state['yo'])**2)
            if dist > 1.0: reward += 0.1 
        else:
            reward = -500.0
            return self._get_obs(), reward, True, False, self._get_info()

        done = self.state['t'] >= (self.max_episode_steps * self.constants['ep'])
        return self._get_obs(), reward, False, done, self._get_info()

    def _ode_func(self, t, y):
        v, a, dx, dy, w = y[2], y[3], y[4], y[5], y[6]
        actual_a = a if (v > 1e-5 or a > 0) else 0
        return [v*dx, v*dy, actual_a, 0, -w*dy, w*dx, 0, 0, 0, 0, 1]

    def _check_safety(self) -> bool:
        x, y, v = self.state['x'], self.state['y'], self.state['v']
        xo, yo = self.state['xo'], self.state['yo']
        stop_d = self.functions['stopDist'](v)
        return (abs(x - xo) > stop_d) or (abs(y - yo) > stop_d)

    def _get_obs(self):
        rel_x = self.state['x'] - self.state['xo']
        rel_y = self.state['y'] - self.state['yo']
        base_obs = [self.state[k] for k in self._state_keys]
        return np.array(base_obs + [rel_x, rel_y], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self._solve_initial_state_with_z3()
        return self._get_obs(), self._get_info()

    def _solve_initial_state_with_z3(self):
        s = z3.Solver()
        z = {k: z3.Real(k) for k in self._state_keys}
        s.add(z['v'] == 0, z['dx'] == 1, z['dy'] == 0) # Simplified direction for testing
        s.add(z3.Abs(z['x'] - z['xo']) > 20) 
        if s.check() == z3.sat:
            m = s.model()
            res = {}
            for k in self._state_keys:
                val = m[z[k]]
                res[k] = float(val.as_decimal(10).replace('?', '')) if val is not None else 0.0
            return res
        return {k: 0.0 for k in self._state_keys}

    def _get_info(self): return self.state.copy()

    def _execute_action_config(self, config, agent_action):
        if isinstance(config, ChoiceActionConfig):
            idx = agent_action[f"choice_{config.level_name}"]
            return self._execute_action_config(config.choices[idx], agent_action)
        elif isinstance(config, SequenceActionConfig):
            for a in config.actions:
                r, t, s = self._execute_action_config(a, agent_action)
                if not s: return r, t, s
            return 0, False, True
        elif isinstance(config, DeterministicAssignmentActionConfig):
            val = self.constants['A'] if config.value == 'A' else -self.constants['b']
            self.state[config.variable] = float(val)
            return 0, False, True
        elif isinstance(config, ActionConstraintConfig):
            met = self._evaluate_expression(config.formula)
            return (0, False, True) if met else (-5, False, False)
        return 0, False, False

    def _evaluate_expression(self, expr):
        # Fix: Replace logical '|' with 'or' for Python eval
        clean = expr.replace('v == 0', "self.state['v'] <= 1e-5")
        clean = clean.replace('abs', 'np.abs').replace('|', ' or ')
        for k in self._state_keys: 
            clean = re.sub(r'\b' + k + r'\b', f"self.state['{k}']", clean)
        for f in self.functions: 
            clean = re.sub(r'\b' + f + r'\(', f"self.functions['{f}'](", clean)
        try: 
            return eval(clean, {"self": self, "np": np})
        except Exception as e: 
            return False

    def render(self): pass

def test_steady_movement():
    print("--- Testing Steady Movement Logic ---")
    env = GeneratedKXEnv()
    
    for ep in range(1, 4):
        print(f"\nStarting Episode {ep}")
        obs, info = env.reset()
        total_reward = 0
        
        for i in range(1, 51):
            # Force Choice 2 (Accel)
            action = env.action_space.sample()
            action['choice_ChoiceLevel_0'] = 2 
            
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            
            if i % 10 == 0:
                print(f"Step {i} | Vel: {info['v']:.2f} | Accel: {info['a']:.1f} | Reward: {reward:.2f}")
            
            if terminated or truncated:
                print("Episode ended early.")
                break
        print(f"Episode {ep} Finished. Total Reward: {total_reward:.2f}")

if __name__ == "__main__":
    test_steady_movement()

--- Testing Steady Movement Logic ---

Starting Episode 1
Step 10 | Vel: 5.00 | Accel: 5.0 | Reward: 8.10
Step 20 | Vel: 10.00 | Accel: 5.0 | Reward: 15.60
Step 30 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Step 40 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Step 50 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Episode 1 Finished. Total Reward: 62.95

Starting Episode 2
Step 10 | Vel: 5.00 | Accel: 5.0 | Reward: 8.10
Step 20 | Vel: 10.00 | Accel: 5.0 | Reward: 15.60
Step 30 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Step 40 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Step 50 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Episode 2 Finished. Total Reward: 62.95

Starting Episode 3
Step 10 | Vel: 5.00 | Accel: 5.0 | Reward: 8.10
Step 20 | Vel: 10.00 | Accel: 5.0 | Reward: 15.60
Step 30 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Step 40 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Step 50 | Vel: 11.00 | Accel: 5.0 | Reward: -5.00
Episode 3 Finished. Total Reward: 62.95


In [16]:
import gymnasium as gym

def test_environment():
    # 1. Initialize Environment
    print("Initializing Environment...")
    env = GeneratedKXEnv()
    
    # 2. Test Reset (Triggers Z3 Solver)
    print("Testing Reset...")
    obs, info = env.reset()
    print(f"Initial Observation: {obs}")
    print(f"Initial State Keys: {info.keys()}")

    # 3. Run a few steps with Random Actions
    print("\nRunning Simulation...")
    for i in range(10):
        # Sample a random action from the space
        action = env.action_space.sample()
        
        # Step the environment
        obs, reward, terminated, truncated, info = env.step(action)
        
        print(f"Step {i+1}:")
        print(f"  Action Chosen: Choice {action['choice_ChoiceLevel_0']}")
        print(f"  Reward: {reward}")
        print(f"  CISB Buffer Size: {len(env.cisb_buffer)}")
        
        if terminated or truncated:
            print("  Episode Finished. Resetting...")
            obs, info = env.reset()

    print("\nTest Complete!")
    env.close()

if __name__ == "__main__":
    test_environment()

Initializing Environment...
Testing Reset...
Initial Observation: [21.  0.  0.  0.  1.  0.  0.  0.  0.  0.  0. 21.  0.]
Initial State Keys: dict_keys(['x', 'y', 'v', 'a', 'dx', 'dy', 'w', 'r', 'xo', 'yo', 't'])

Running Simulation...
Step 1:
  Action Chosen: Choice 0
  Reward: -0.9


AttributeError: 'GeneratedKXEnv' object has no attribute 'cisb_buffer'

In [17]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO

class SB3ActionWrapper(gym.ActionWrapper):
    def __init__(self, env):
        super().__init__(env)
        # We redefine the action space as a simple Discrete space (0, 1, 2)
        # This matches your 'choice_ChoiceLevel_0'
        self.action_space = gym.spaces.Discrete(3)

    def action(self, action):
        # This method takes the integer from SB3 and 
        # converts it back into the Dict the environment needs
        return {
            'choice_ChoiceLevel_0': int(action),
            'assign_r': np.array(1.0, dtype=np.float32),  # Default values
            'assign_w': np.array(0.0, dtype=np.float32),
            'assign_xo': np.array(0.0, dtype=np.float32),
            'assign_yo': np.array(0.0, dtype=np.float32)
        }

# 1. Initialize Environment
raw_env = GeneratedKXEnv()

# 2. Wrap it so SB3 can read it
env = SB3ActionWrapper(raw_env)

# 3. Initialize the Agent
# Now SB3 sees a Discrete(3) action space and is happy
model = PPO(
    "MlpPolicy", 
    env, 
    verbose=1, 
    learning_rate=0.0003,
)

# 4. Start Training
print("Starting training...")
model.learn(total_timesteps=50000)

# 5. Save and Test
model.save("ppo_kx_model")
print("Training complete!")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Starting training...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 155      |
|    ep_rew_mean     | 940      |
| time/              |          |
|    fps             | 1614     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 148         |
|    ep_rew_mean          | 1.1e+03     |
| time/                   |             |
|    fps                  | 1187        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014847156 |
|    clip_fraction        | 0.294       |
|    clip_range           | 0.2         |

In [18]:
import time
import numpy as np

def test_trained_agent(env, model, num_episodes=1):
    for episode in range(num_episodes):
        # FIX: Unpack the tuple. SB3's predict() needs the array, not the (obs, info) tuple.
        obs, info = env.reset() 
        
        done = False
        score = 0
        step = 0
        
        print(f"\n--- Starting Test Episode {episode + 1} ---")
        print(f"{'Step':<5} | {'Pos (x)':<8} | {'Vel (v)':<8} | {'Action':<10} | {'Reward':<8} | {'Safety Status'}")
        print("-" * 70)

        while not done:
            # Now 'obs' is a clean numpy array
            action, _states = model.predict(obs, deterministic=True)
            
            # FIX: Gymnasium step() returns 5 values (obs, reward, terminated, truncated, info)
            # SB3 VecEnv step() returns 4. Since you are using a wrapped env, 
            # ensure you handle the return correctly.
            result = env.step(action)
            
            # Handling potential 4 or 5 value returns depending on your specific wrapper
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            
            # Extract values for display
            x_pos = obs[0]
            velocity = obs[2]
            
            # Visual Track Logic
            track_length = 20
            pos_index = int(min(max(x_pos, 0), 20)) 
            visual_track = ["."] * (track_length + 1)
            if 0 <= pos_index <= track_length:
                visual_track[pos_index] = "A"
            track_str = "".join(visual_track)

            action_names = {0: "BRAKE", 1: "COAST", 2: "ACCEL"}
            a_name = action_names.get(int(action), str(action))
            safety_status = "SAFE" if reward > 0 else "VIOLATION!"

            print(f"{step:<5} | {x_pos:<8.2f} | {velocity:<8.2f} | {a_name:<10} | {reward:<8.1f} | [{track_str}] {safety_status}")
            
            score += reward
            step += 1
            time.sleep(0.05)

        print("-" * 70)
        print(f"Episode Finished. Total Score: {score}")

# Run the fixed test
test_trained_agent(env, model)


--- Starting Test Episode 1 ---
Step  | Pos (x)  | Vel (v)  | Action     | Reward   | Safety Status
----------------------------------------------------------------------
0     | 21.02    | 0.50     | ACCEL      | 1.4      | [....................A] SAFE
1     | 21.10    | 1.00     | ACCEL      | 2.1      | [....................A] SAFE
2     | 21.23    | 1.50     | ACCEL      | 2.9      | [....................A] SAFE
3     | 21.40    | 2.00     | ACCEL      | 3.6      | [....................A] SAFE
4     | 21.62    | 2.50     | ACCEL      | 4.3      | [....................A] SAFE
5     | 21.90    | 3.00     | ACCEL      | 5.1      | [....................A] SAFE
6     | 22.23    | 3.50     | ACCEL      | 5.8      | [....................A] SAFE
7     | 22.60    | 4.00     | ACCEL      | 6.6      | [....................A] SAFE
8     | 23.02    | 4.50     | ACCEL      | 7.3      | [....................A] SAFE
9     | 23.50    | 5.00     | ACCEL      | 8.1      | [....................A] SAF

KeyboardInterrupt: 